In [2]:
import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)
print("Project 2 environment ready")

DuckDB: 2.0.0
Pandas: 3.1.1
Project 2 environment ready


In [1]:

import duckdb
from pathlib import Path

# Actual Project 1 database location
project1_db = Path(
    "/Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-demand-sensing-planner/data/warehouse/demand_intelligence.duckdb"
)

# Verify the database exists
print("Database exists:", project1_db.is_file())

if not project1_db.is_file():
    raise FileNotFoundError(f"Database not found: {project1_db}")

# Connect in read-only mode
con = duckdb.connect(
    str(project1_db),
    read_only=True
)

print("Successfully connected to Project 1!")

# Retrieve available reporting views
views = con.sql("""
    SELECT table_schema, table_name
    FROM information_schema.views
    WHERE table_schema NOT IN (
        'information_schema',
        'pg_catalog'
    )
    ORDER BY table_schema, table_name
""").df()

print("Number of views:", len(views))
display(views)


Database exists: True
Successfully connected to Project 1!
Number of views: 30


,table_schema,table_name
0,main,duckdb_columns
1,main,duckdb_constraints
2,main,duckdb_databases
3,main,duckdb_indexes
4,main,duckdb_logs
5,main,duckdb_schemas
6,main,duckdb_tables
7,main,duckdb_types
8,main,duckdb_views
9,main,pragma_database_list


In [2]:
# STEP 4: Inspect Project 1 demand and forecasting views

candidate_views = [
    "v_airpods_weekly_demand",
    "v_forecast_lead_time_baseline",
    "v_forecast_model_performance",
    "v_inventory_weekly"
]

for view_name in candidate_views:

    print(f"\n{'=' * 65}")
    print(f"VIEW: reporting.{view_name}")
    print(f"{'=' * 65}")

    # Inspect column names and data types
    schema = con.execute(
        f"DESCRIBE SELECT * FROM reporting.{view_name}"
    ).df()

    print("\nCOLUMN STRUCTURE:")
    display(schema)

    # Preview the first five records
    preview = con.execute(
        f"SELECT * FROM reporting.{view_name} LIMIT 5"
    ).df()

    print("\nSAMPLE DATA:")
    display(preview)



VIEW: reporting.v_airpods_weekly_demand

COLUMN STRUCTURE:


,column_name,column_type,null,key,default,extra
0,Week_Ending,DATE,YES,None,None,None
1,Units_Sold,DOUBLE,YES,None,None,None



SAMPLE DATA:


,Week_Ending,Units_Sold
0,2019-01-13,187.0
1,2019-01-20,184.0
2,2019-01-27,175.0
3,2019-02-03,225.0
4,2019-02-10,247.0



VIEW: reporting.v_forecast_lead_time_baseline

COLUMN STRUCTURE:


,column_name,column_type,null,key,default,extra
0,Forecast_Origin,DATE,YES,None,None,None
1,Forecast_Week,DATE,YES,None,None,None
2,Lead_Time_Weeks,BIGINT,YES,None,None,None
3,Model,VARCHAR,YES,None,None,None
4,Forecast_Weekly_Units,DOUBLE,YES,None,None,None
5,Cumulative_Forecast_Units,DOUBLE,YES,None,None,None
6,Scenario_Type,VARCHAR,YES,None,None,None


... (6 outputs left out)


VIEW: reporting.v_inventory_weekly

COLUMN STRUCTURE:


,column_name,column_type,null,key,default,extra
0,Scenario,VARCHAR,YES,None,None,None
1,Week,BIGINT,YES,None,None,None
2,Forecast_Week,VARCHAR,YES,None,None,None
3,Beginning_Inventory,BIGINT,YES,None,None,None
4,Inbound_Receipts,BIGINT,YES,None,None,None
5,Demand_Units,BIGINT,YES,None,None,None
6,Fulfilled_Units,BIGINT,YES,None,None,None
7,Shortage_Units,BIGINT,YES,None,None,None
8,Ending_Inventory,BIGINT,YES,None,None,None
9,Weekly_Fill_Rate_Pct,DOUBLE,YES,None,None,None



SAMPLE DATA:


,Scenario,Week,Forecast_Week,Beginning_Inventory,Inbound_Receipts,Demand_Units,Fulfilled_Units,Shortage_Units,Ending_Inventory,Weekly_Fill_Rate_Pct
0,Baseline,1,2020-01-05,1400,0,515,515,0,885,100.0
1,Baseline,2,2020-01-12,885,0,515,515,0,370,100.0
2,Baseline,3,2020-01-19,370,900,515,515,0,755,100.0
3,Baseline,4,2020-01-26,755,0,515,515,0,240,100.0
4,Demand Surge,1,2020-01-05,1400,0,644,644,0,756,100.0


In [3]:
# STEP 5A: Extract and audit Project 1 forecast

import pandas as pd

forecast = con.execute("""
    SELECT
        Forecast_Origin,
        Forecast_Week,
        Lead_Time_Weeks,
        Model,
        Forecast_Weekly_Units,
        Scenario_Type
    FROM reporting.v_forecast_lead_time_baseline
    ORDER BY Forecast_Origin, Lead_Time_Weeks
""").df()

print("Total forecast rows:", len(forecast))
print("Forecast origins:", forecast["Forecast_Origin"].nunique())
print("Maximum horizon:", forecast["Lead_Time_Weeks"].max())
print("Forecast models:", forecast["Model"].unique())
print("Scenario types:", forecast["Scenario_Type"].unique())

print("\nMissing values:")
display(forecast.isna().sum())

print("\nFULL FORECAST:")
display(forecast)


Total forecast rows: 4
Forecast origins: 1
Maximum horizon: 4
Forecast models: <StringArray>
['Naive Forecast']
Length: 1, dtype: str
Scenario types: <StringArray>
['Historical Illustrative Baseline']
Length: 1, dtype: str

Missing values:


Forecast_Origin          0
Forecast_Week            0
Lead_Time_Weeks          0
Model                    0
Forecast_Weekly_Units    0
Scenario_Type            0
dtype: int64


FULL FORECAST:


,Forecast_Origin,Forecast_Week,Lead_Time_Weeks,Model,Forecast_Weekly_Units,Scenario_Type
0,2019-12-29,2020-01-05,1,Naive Forecast,515.0,Historical Illustrative Baseline
1,2019-12-29,2020-01-12,2,Naive Forecast,515.0,Historical Illustrative Baseline
2,2019-12-29,2020-01-19,3,Naive Forecast,515.0,Historical Illustrative Baseline
3,2019-12-29,2020-01-26,4,Naive Forecast,515.0,Historical Illustrative Baseline


In [4]:
# STEP 5B: Export the original Project 1 forecast

from pathlib import Path

repo_root = Path.cwd()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

output_dir = repo_root / "data" / "raw" / "project1"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "project1_forecast_export.csv"

assert not forecast.empty, "Forecast data is empty"
assert forecast["Forecast_Weekly_Units"].notna().all()
assert (forecast["Forecast_Weekly_Units"] >= 0).all()

forecast.to_csv(output_file, index=False)

print("Forecast exported successfully!")
print("File:", output_file)
print("Rows:", len(forecast))

display(forecast.head(10))


Forecast exported successfully!
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/raw/project1/project1_forecast_export.csv
Rows: 4


,Forecast_Origin,Forecast_Week,Lead_Time_Weeks,Model,Forecast_Weekly_Units,Scenario_Type
0,2019-12-29,2020-01-05,1,Naive Forecast,515.0,Historical Illustrative Baseline
1,2019-12-29,2020-01-12,2,Naive Forecast,515.0,Historical Illustrative Baseline
2,2019-12-29,2020-01-19,3,Naive Forecast,515.0,Historical Illustrative Baseline
3,2019-12-29,2020-01-26,4,Naive Forecast,515.0,Historical Illustrative Baseline


In [5]:
# STEP 6A: Build 12-week demand planning horizon

import pandas as pd

# Copy the validated Project 1 forecast
demand = forecast.copy()

# Convert forecast dates
demand["Forecast_Week"] = pd.to_datetime(
    demand["Forecast_Week"]
)

# Sort chronologically
demand = demand.sort_values(
    "Forecast_Week"
).reset_index(drop=True)

# Validate the four original forecast weeks
assert len(demand) == 4
assert demand["Forecast_Week"].is_unique
assert demand["Forecast_Weekly_Units"].notna().all()
assert (demand["Forecast_Weekly_Units"] >= 0).all()
assert demand["Forecast_Week"].diff().dropna().eq(
    pd.Timedelta(weeks=1)
).all()

# Set up the original four-week forecast
original = pd.DataFrame({
    "Week": range(1, 5),
    "Week_Start": demand["Forecast_Week"],
    "SKU": "AIRPODS_REFERENCE",
    "Demand_Units": demand["Forecast_Weekly_Units"],
    "Demand_Source": "Project1_Original_Forecast"
})

# Extend the planning horizon to 12 weeks
last_date = original["Week_Start"].max()
last_demand = original["Demand_Units"].iloc[-1]

extension = pd.DataFrame({
    "Week": range(5, 13),
    "Week_Start": pd.date_range(
        start=last_date + pd.Timedelta(weeks=1),
        periods=8,
        freq="7D"
    ),
    "SKU": "AIRPODS_REFERENCE",
    "Demand_Units": last_demand,
    "Demand_Source": "Flat_Extension_Assumption"
})

# Combine original forecast and extension
demand_plan = pd.concat(
    [original, extension],
    ignore_index=True
)

demand_plan["Scenario"] = "Baseline"

print("12-week demand plan created!")
print("Planning weeks:", len(demand_plan))
print("Total planned demand:",
      demand_plan["Demand_Units"].sum())

display(demand_plan)


12-week demand plan created!
Planning weeks: 12
Total planned demand: 6180.0


,Week,Week_Start,SKU,Demand_Units,Demand_Source,Scenario
0,1,2020-01-05,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
1,2,2020-01-12,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
2,3,2020-01-19,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
3,4,2020-01-26,AIRPODS_REFERENCE,515.0,Project1_Original_Forecast,Baseline
4,5,2020-02-02,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
5,6,2020-02-09,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
6,7,2020-02-16,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
7,8,2020-02-23,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
8,9,2020-03-01,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline
9,10,2020-03-08,AIRPODS_REFERENCE,515.0,Flat_Extension_Assumption,Baseline


In [6]:
# STEP 6B: Validate and export 12-week demand plan

from pathlib import Path

# Validate completeness and integrity
assert len(demand_plan) == 12
assert demand_plan["Week_Start"].is_unique
assert demand_plan["Demand_Units"].notna().all()
assert (demand_plan["Demand_Units"] >= 0).all()

assert (
    demand_plan["Demand_Source"]
    == "Project1_Original_Forecast"
).sum() == 4

assert (
    demand_plan["Demand_Source"]
    == "Flat_Extension_Assumption"
).sum() == 8

# Locate Project 2 repository
repo_root = Path.cwd()

if repo_root.name == "notebooks":
    repo_root = repo_root.parent

# Create output directory
output_dir = repo_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "demand_plan_12_weeks.csv"

# Save dataset
demand_plan.to_csv(output_file, index=False)

print("SUCCESS: 12-week demand plan exported")
print("File:", output_file)
print("Rows:", len(demand_plan))
print("Total demand:", demand_plan["Demand_Units"].sum())
print("Validation checks: PASSED")


SUCCESS: 12-week demand plan exported
File: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/data/processed/demand_plan_12_weeks.csv
Rows: 12
Total demand: 6180.0
Validation checks: PASSED


## Demand Integration — Final Summary

### Objective
Connect Project 1's demand forecasting outputs to
Project 2's supply network optimization model.

### Data Source
Project 1: Hardware Demand-Sensing & Launch Readiness Planner

Source view:
reporting.v_forecast_lead_time_baseline

### Planning Assumptions
- Product: AIRPODS_REFERENCE
- Planning horizon: 12 weeks
- Original forecast: Weeks 1–4
- Assumed flat extension: Weeks 5–12
- Weekly demand: 515 units
- Total planned demand: 6,180 units
- Demand scenario: Baseline

### Limitations
The first four weeks originate from Project 1's
historical naive forecasting model.

Weeks 5–12 are flat-demand planning assumptions,
not independently generated forecasts.

Supplier capacity, manufacturing constraints,
inventory policies, and transportation decisions
will be introduced in Notebook 02 and later stages.

### Deliverables
1. data/raw/project1/project1_forecast_export.csv
2. data/processed/demand_plan_12_weeks.csv

### Outcome
A validated demand-planning dataset is available
for use in the supply network optimization model.
